# Process monitoring

> Poll running processes and record process start and stop events for a monitoring session.

This module provides the application’s secondary process-activity signal. It periodically compares the current set of running processes with the set observed during the previous polling interval, then records processes that appeared or disappeared.

Process events support debugging and future monitoring features. Foreground-window tracking remains the application’s primary source for attention-time and productivity reporting.


In [ ]:
#| default_exp process_monitor

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import psutil
from datetime import datetime
import time

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import snooper_pkg.config as cf
from snooper_pkg.db import *

## Process snapshots

In [ ]:
#| export
def get_current_processes(excluded_process_set):
    """Return running `(pid, process_name)` pairs excluding configured processes."""
    cur_procs = [tuple(proc.info.values()) for proc in psutil.process_iter(['pid', 'name'])]
    return set([(pid, proc) for (pid, proc) in cur_procs if proc not in excluded_process_set])

## Process event monitoring


In [ ]:
#| export
def start_process_monitoring(session_id, stop_event, excluded_processes, time_interval=cf.PROCESS_MONITORING_INTERVAL_SECONDS):
    """Poll processes until stopped and log process start and stop events."""
    t0_procs = get_current_processes(excluded_processes)
    while not stop_event.is_set():
        try:
            time.sleep(time_interval)
            t1_procs = get_current_processes(excluded_processes)
            opened_procs = t1_procs - t0_procs
            closed_procs = t0_procs - t1_procs
            cur_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            t0_procs = t1_procs
            if opened_procs:
                for p in opened_procs: log_process_event(session_id, p[0], p[1], 'start', cur_time)
                print(f"Opened processes: {cur_time}-{opened_procs}")
            if closed_procs:
                for p in closed_procs: log_process_event(session_id, p[0], p[1], 'stop', cur_time)
                print(f"Closed processes: {cur_time}-{closed_procs}")
        except KeyboardInterrupt:
            break


NameError: name 'cf' is not defined

- `get_current_processes` assumes the values returned by `proc.info` retain the requested `['pid', 'name']` order. Confirm this is an intentional `psutil` usage assumption.
- The module logs process events and prints changes, but does not document or handle process-access exceptions such as permissions errors. Confirm whether those cases are handled elsewhere or should be addressed after the nbdev migration.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()